# **MediQuery — A Retrieval-Augmented Generation (RAG) Medical Information Assistant**

MediQuery answers general health questions by **retrieving** relevant passages from a curated knowledge base and then **grounding** a language model's answer in that retrieved text, instead of letting the model answer purely from what it memorized during training.

> ⚠️ **Disclaimer**: MediQuery is a portfolio / learning project. It provides **general educational information only** — it is not a diagnostic tool, is not reviewed by medical professionals, and must never replace advice from a qualified doctor. Every answer it gives ends with a reminder of this.

## **RAG Intuition**

A plain LLM answers from what it learned during training — it can be outdated, generic, or simply wrong (a "hallucination"), and it can't tell you where an answer came from.

**Main Idea**

Retrieval-Augmented Generation (RAG) fixes this by giving the model an *open-book exam* instead of a closed one:

1. Store trusted documents as vectors (embeddings) in a searchable index.
2. When a question comes in, **retrieve** the most relevant chunks of text.
3. **Augment** the model's prompt with those chunks.
4. The model **generates** an answer grounded in the retrieved text — and can cite where it came from.

**Why RAG fits a medical assistant particularly well**

* The knowledge base can be updated (new guidelines, corrected facts) without retraining the model.
* Answers can be traced back to a source document, which matters a lot for health information.
* The model is discouraged from inventing facts, because it's instructed to only use the retrieved context.

**Key Components**

1. Knowledge Base → the trusted documents
2. Embedding model → turns text into vectors
3. Vector store (FAISS) → fast similarity search over those vectors
4. Retriever → pulls the top-k most relevant chunks for a query
5. Generator (LLM) → writes the final answer using only the retrieved chunks
6. Safety layer → disclaimers + emergency-keyword detection

### **Hands-on: Building MediQuery**

Install Libraries

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate langchain langchain-community gradio

Import Libraries

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter

### **Step 1: Build the Medical Knowledge Base**

For this demo the knowledge base is a small, hand-written set of general health overviews (symptoms + general self-care pointers, no dosages, no diagnosis). In a real deployment this would instead be built from vetted sources such as WHO fact sheets, CDC pages, or MedlinePlus, reviewed by a clinician before being indexed.

In [ ]:
knowledge_base = [
    {
        "id": "common_cold",
        "title": "Common Cold",
        "text": "The common cold is a mild viral infection of the nose and throat. "
                "Typical symptoms include a runny or stuffy nose, sneezing, sore throat, "
                "mild cough, and low energy, usually lasting seven to ten days. "
                "Rest, fluids, and over-the-counter symptom relief are generally enough. "
                "See a doctor if symptoms last more than ten days or a high fever develops."
    },
    {
        "id": "influenza",
        "title": "Influenza (Flu)",
        "text": "Influenza is a contagious respiratory illness caused by flu viruses. "
                "It tends to come on suddenly with fever, chills, body aches, fatigue, "
                "headache, and a dry cough, and is usually more severe than a common cold. "
                "Most people recover within one to two weeks with rest and fluids. "
                "Older adults, young children, and people with chronic conditions face a "
                "higher risk of complications and should seek medical care promptly."
    },
    {
        "id": "type2_diabetes",
        "title": "Type 2 Diabetes",
        "text": "Type 2 diabetes is a chronic condition in which the body does not use "
                "insulin properly, leading to high blood sugar. Common symptoms include "
                "increased thirst, frequent urination, fatigue, and blurred vision, though "
                "many people have no symptoms early on. Management generally involves diet, "
                "physical activity, weight management, and medical monitoring of blood sugar. "
                "It is diagnosed and managed by a healthcare provider through blood tests."
    },
    {
        "id": "hypertension",
        "title": "Hypertension (High Blood Pressure)",
        "text": "Hypertension means the force of blood against artery walls is consistently "
                "too high. It is often called a silent condition because it usually has no "
                "obvious symptoms, though very high readings can cause headaches or "
                "shortness of breath. Lifestyle factors such as diet, salt intake, exercise, "
                "stress, and weight all influence blood pressure. Regular monitoring by a "
                "healthcare professional is the main way it is detected and managed."
    },
    {
        "id": "asthma",
        "title": "Asthma",
        "text": "Asthma is a chronic condition in which the airways become inflamed and "
                "narrowed, making breathing difficult. Common symptoms include wheezing, "
                "shortness of breath, chest tightness, and coughing, often triggered by "
                "allergens, exercise, or cold air. Long-term management usually involves "
                "identifying triggers and following a treatment plan set by a doctor. "
                "Sudden, severe breathing difficulty is a medical emergency."
    },
    {
        "id": "migraine",
        "title": "Migraine",
        "text": "A migraine is a neurological condition causing recurring, often severe "
                "headaches, frequently on one side of the head, sometimes with nausea and "
                "sensitivity to light or sound. Attacks can last from a few hours to a few "
                "days. Common self-care steps include resting in a quiet, dark room and "
                "identifying personal triggers such as stress, certain foods, or poor sleep. "
                "A doctor can help with diagnosis and a longer-term management plan."
    },
    {
        "id": "gerd",
        "title": "GERD (Acid Reflux)",
        "text": "Gastroesophageal reflux disease (GERD) occurs when stomach acid "
                "repeatedly flows back into the esophagus, causing heartburn, regurgitation, "
                "and sometimes chest discomfort after eating or when lying down. Avoiding "
                "trigger foods, eating smaller meals, and not lying down right after eating "
                "can help. Frequent or severe symptoms should be evaluated by a doctor, "
                "since they can resemble other conditions."
    },
    {
        "id": "seasonal_allergies",
        "title": "Seasonal Allergies",
        "text": "Seasonal allergies happen when the immune system overreacts to pollen or "
                "other airborne particles, causing sneezing, itchy or watery eyes, a runny "
                "nose, and congestion. Symptoms typically follow the pollen season for "
                "trees, grasses, or weeds in a given region. Reducing exposure and general "
                "symptom relief are common first steps; a doctor or allergist can help with "
                "persistent or severe cases."
    },
    {
        "id": "iron_deficiency_anemia",
        "title": "Iron-Deficiency Anemia",
        "text": "Iron-deficiency anemia occurs when the body lacks enough iron to produce "
                "healthy red blood cells, which can cause fatigue, pale skin, weakness, and "
                "shortness of breath. It can result from inadequate dietary iron, blood loss, "
                "or poor absorption. Diagnosis is done through blood tests, and treatment is "
                "guided by a healthcare provider based on the underlying cause."
    },
    {
        "id": "hypothyroidism",
        "title": "Hypothyroidism",
        "text": "Hypothyroidism occurs when the thyroid gland does not produce enough "
                "thyroid hormone, which can slow the body's metabolism. Common symptoms "
                "include fatigue, weight gain, feeling cold, dry skin, and low mood. "
                "It is diagnosed with blood tests and typically managed long-term under a "
                "doctor's supervision."
    },
    {
        "id": "generalized_anxiety",
        "title": "Generalized Anxiety",
        "text": "Generalized anxiety involves persistent, excessive worry about everyday "
                "things that can be hard to control, along with restlessness, fatigue, "
                "difficulty concentrating, or sleep problems. Many people find relief "
                "through relaxation techniques, regular exercise, and reducing caffeine, "
                "alongside support from a mental health professional for ongoing or "
                "distressing symptoms."
    },
    {
        "id": "lower_back_pain",
        "title": "Lower Back Pain",
        "text": "Lower back pain is commonly caused by muscle strain, poor posture, or "
                "prolonged sitting, and usually improves within a few weeks with gentle "
                "movement, stretching, and avoiding heavy lifting. Applying heat or cold and "
                "staying moderately active (rather than complete bed rest) generally helps "
                "recovery. Pain that is severe, spreads down a leg, or follows an injury "
                "should be checked by a doctor."
    },
]

print(f"Knowledge base loaded: {len(knowledge_base)} documents")

### **Step 2: Chunk the Documents**

These entries are already short, but real medical documents (guidelines, articles) are much longer. `RecursiveCharacterTextSplitter` breaks long text into overlapping chunks so that each retrieved piece is small and focused enough for the model to use well.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)

chunks = []          # the actual text passed to the embedding model
chunk_metadata = []  # which source document each chunk came from

for doc in knowledge_base:
    pieces = splitter.split_text(doc["text"])
    for piece in pieces:
        chunks.append(piece)
        chunk_metadata.append({"id": doc["id"], "title": doc["title"]})

print(f"Total chunks: {len(chunks)}")

### **Step 3: Generate Embeddings**

We use `all-MiniLM-L6-v2` from `sentence-transformers` — small, fast, and a common default for RAG prototypes since it runs comfortably on CPU.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
chunk_embeddings = chunk_embeddings.astype("float32")

print("Embedding matrix shape:", chunk_embeddings.shape)

### **Step 4: Build the Vector Store (FAISS)**

FAISS indexes the embedding vectors so that, given a new query vector, it can quickly find the closest (most similar) chunks — this is the "search" half of retrieval-augmented generation.

In [ ]:
embedding_dim = chunk_embeddings.shape[1]

# Normalize so that inner product search behaves like cosine similarity
faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print("Vectors in index:", index.ntotal)

### **Step 5: Build the Retriever**

In [ ]:
def retrieve(query, k=3):
    query_vec = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)

    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx],
            "title": chunk_metadata[idx]["title"],
            "score": float(score),
        })
    return results

Test Retrieval

In [ ]:
for r in retrieve("What are the symptoms of high blood pressure?", k=3):
    print(f"[{r['title']}] (score={r['score']:.3f})")
    print(r["text"])
    print()

### **Step 6: Add the Generator (LLM)**

We use `google/flan-t5-base`, an instruction-tuned model small enough to run on a free Colab CPU/GPU with no API key required. The generator is a swappable piece of the pipeline — in a production build you could replace `generate_answer()` with a call to any larger LLM API without touching the retrieval logic above.

In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-base", max_new_tokens=200)

### **Step 7: Build the RAG Prompt Template**

The prompt instructs the model to answer **only** from the retrieved context, and to say so plainly when the context doesn't cover the question — this is what keeps a RAG system from quietly making things up.

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(f"- {c['text']}" for c in retrieved_chunks)
    prompt = (
        "You are a medical information assistant. Answer the question using ONLY "
        "the context below. If the context does not contain the answer, say "
        "'I don't have enough information to answer that.' Do not add facts that "
        "are not in the context. Do not give a diagnosis or dosage instructions.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt

### **Step 8: Safety Layer**

Two lightweight guardrails before anything reaches the generator:

1. **Emergency detection** — certain phrases suggest a potential emergency, where the right answer is "contact emergency services now," not a generated paragraph.
2. **Standing disclaimer** — every normal answer is followed by a reminder that this is general information, not medical advice.

In [ ]:
EMERGENCY_KEYWORDS = [
    "chest pain", "can't breathe", "cannot breathe", "severe bleeding",
    "suicidal", "suicide", "want to die", "overdose", "unconscious",
    "stroke symptoms", "not breathing",
]

DISCLAIMER = ("\n\n_This is general educational information, not medical advice. "
              "Please consult a qualified healthcare professional for your specific situation._")

def is_emergency(query):
    q = query.lower()
    return any(keyword in q for keyword in EMERGENCY_KEYWORDS)

def emergency_response():
    return ("This may describe a medical emergency. Please contact your local "
            "emergency number or go to the nearest emergency room right away. "
            "If this is a mental health crisis, please reach out to a crisis "
            "helpline or emergency services immediately — you don't have to "
            "handle this alone.")

### **Step 9: Full RAG Pipeline — MediQuery Assistant**

Retrieve → build prompt → generate → attach sources and disclaimer.

In [ ]:
def mediquery_answer(query, k=3):
    if is_emergency(query):
        return {"answer": emergency_response(), "sources": []}

    retrieved = retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)
    raw_answer = generator(prompt)[0]["generated_text"]

    return {
        "answer": raw_answer.strip() + DISCLAIMER,
        "sources": sorted({r["title"] for r in retrieved}),
    }

### **Step 10: Try It Out**

In [ ]:
test_queries = [
    "What are common symptoms of the flu?",
    "How is type 2 diabetes usually managed?",
    "What can help with a migraine?",
    "I have chest pain, what should I do?",
]

for q in test_queries:
    result = mediquery_answer(q)
    print("Q:", q)
    print("A:", result["answer"])
    print("Sources:", result["sources"])
    print("-" * 60)

### **Step 11 (Optional): Interactive UI with Gradio**

A minimal chat-style front end so MediQuery can be demoed as a live app rather than just notebook cells — useful for a portfolio walkthrough or a recorded demo video.

In [ ]:
import gradio as gr

def gradio_chat(message, history):
    result = mediquery_answer(message)
    return result["answer"]

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="MediQuery — RAG Medical Information Assistant",
    description="General health information, grounded in a small curated knowledge base. Not a substitute for professional medical advice.",
)

demo.launch(debug=False)

### **Future Improvements**

* Replace the hand-written knowledge base with a larger, vetted medical corpus (e.g. WHO fact sheets, MedlinePlus, CDC pages) reviewed by a clinician before indexing.
* Swap `flan-t5-base` for a stronger model via an API (OpenAI, Anthropic, or a hosted open-weight model) — only `generate_answer` needs to change.
* Add re-ranking of retrieved chunks (e.g. a cross-encoder) before generation for higher precision.
* Evaluate answer quality and faithfulness with a framework such as RAGAS.
* Add conversational memory so MediQuery can handle follow-up questions.
* Expand the safety layer: broaden emergency-keyword coverage, add a toxicity/PII filter, and log low-confidence retrievals for human review.
* Wrap the pipeline in a FastAPI service for deployment, with the Gradio/React UI as a client.

---
**MediQuery is a demonstration of RAG architecture built for a portfolio project. It is not a certified medical device, has not been clinically validated, and must not be used for real diagnosis or treatment decisions.**